In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


# Load data
df = pd.read_csv('creditcard.csv')

# Split
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [ ]:
# SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_sm.value_counts().to_dict())

In [5]:
lr = LogisticRegression(max_iter=1000 ,random_state=42)
lr.fit(X_train_sm, y_train_sm)
y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test,y_pred_lr))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     56864
           1       0.12      0.90      0.20        98

    accuracy                           0.99     56962
   macro avg       0.56      0.94      0.60     56962
weighted avg       1.00      0.99      0.99     56962



C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
rf=RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1,class_weight='balanced')
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)
print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from xgboost import XGBClassifier


xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train_sm, y_train_sm)
y_pred_xgb = xgb.predict(X_test)

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
import joblib

joblib.dump(rf, 'fraud_model.pkl')
print("Model saved ✅")

In [ ]:
# Real fraud aur normal transactions nikalo
fraud_sample = df[df['Class'] == 1].iloc[0]
normal_sample = df[df['Class'] == 0].iloc[0]

print("=== FRAUD SAMPLE ===")
print(fraud_sample.drop('Class'))

print("\n=== NORMAL SAMPLE ===")
print(normal_sample.drop('Class'))

In [ ]:
normal_cases = df[df['Class'] == 0]
row = normal_cases.iloc[0]
pred = rf.predict([row.drop('Class').values])
print(f"Predicted = {pred[0]}, Actual = 0")

In [ ]:
# 10 fraud + 20 normal transactions nikalo
fraud_test = df[df['Class'] == 1].head(10)
normal_test = df[df['Class'] == 0].head(20)

# Combine karo
test_csv = pd.concat([fraud_test, normal_test])

# Class column hata do — user nahi jaanta
test_csv = test_csv.drop('Class', axis=1)

# Save karo
test_csv.to_csv('sample_transactions.csv', index=False)
print("Test CSV saved ✅")
print(f"Total transactions: {len(test_csv)}")

In [9]:
y_test.value_counts()

Class
0    56864
1       98
Name: count, dtype: int64

In [10]:
# Test dataset save 
X_test_save = X_test.copy()
X_test_save['Class'] = y_test.values

# Save in csv
X_test_save.to_csv('test_transactions.csv', index=False)
print("Test CSV saved ✅")
print(f"Total: {len(X_test_save)}")
print(f"Fraud: {y_test.sum()}")
print(f"Normal: {len(y_test) - y_test.sum()}")

Test CSV saved ✅
Total: 56962
Fraud: 98
Normal: 56864


In [13]:
X_test_save = X_test.copy()
# X_test_save['Class'] = y_test.values

X_test_save.to_csv('test_transactions.csv', index=False)
print("Saved ✅")
print(f"Total: {len(X_test_save)}")
print(f"Fraud: {y_test.sum()}")
print(f"Normal: {len(y_test) - y_test.sum()}")

Saved ✅
Total: 56962
Fraud: 98
Normal: 56864


In [14]:
X_test_save

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
263020,160760.0,-0.674466,1.408105,-1.110622,-1.328366,1.388996,-1.308439,1.885879,-0.614233,0.311652,...,0.394322,0.080084,0.810034,-0.224327,0.707899,-0.135837,0.045102,0.533837,0.291319,23.00
11378,19847.0,-2.829816,-2.765149,2.537793,-1.074580,2.842559,-2.153536,-1.795519,-0.250020,3.073504,...,-0.515765,-0.295555,0.109305,-0.813272,0.042996,-0.027660,-0.910247,0.110802,-0.511938,11.85
147283,88326.0,-3.576495,2.318422,1.306985,3.263665,1.127818,2.865246,1.444125,-0.718922,1.874046,...,2.034786,-1.060151,0.016867,-0.132058,-1.483996,-0.296011,0.062823,0.552411,0.509764,76.07
219439,141734.0,2.060386,-0.015382,-1.082544,0.386019,-0.024331,-1.074935,0.207792,-0.338140,0.455091,...,-0.192024,-0.281684,-0.639426,0.331818,-0.067584,-0.283675,0.203529,-0.063621,-0.060077,0.99
36939,38741.0,1.209965,1.384303,-1.343531,1.763636,0.662351,-2.113384,0.854039,-0.475963,-0.629658,...,0.009083,-0.164015,-0.328294,-0.154631,0.619449,0.818998,-0.330525,0.046884,0.104527,1.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54199,46329.0,-0.863057,0.225429,2.458855,0.613099,0.067149,1.716173,-0.254484,0.825754,0.407484,...,-0.207556,0.183183,0.695818,0.101555,-0.683590,-0.505613,-0.343860,0.218972,0.127074,35.97
184616,126310.0,1.397799,-1.426153,-0.369131,0.891825,-1.164153,-0.166657,-0.320745,-0.023070,1.800818,...,0.340394,-0.058383,-0.479606,0.053303,-0.117313,-0.425722,-0.568239,0.001064,0.017812,297.63
274532,166070.0,-1.047727,0.685141,0.195457,-3.583402,0.082922,-0.444060,0.261275,0.457403,1.225167,...,-0.222906,-0.120079,-0.388817,-0.095996,-1.059673,-0.392663,-0.781760,0.017347,0.145133,16.39
269819,163789.0,2.159972,-1.084234,-0.858819,-1.126188,-0.647032,0.234289,-1.164932,0.138244,-0.045273,...,0.024393,0.327882,0.932738,0.077597,0.119508,-0.090098,-0.101767,-0.002565,-0.056766,19.95


In [15]:

fraud_only = X_test[y_test == 1].copy()
fraud_only['Class'] = 1

fraud_only.to_csv('fraud_only.csv', index=False)
print(f"Fraud transactions: {len(fraud_only)}")

Fraud transactions: 98
